# 02 — Analysis: Figure 1, Table 1, sanity checksAttach every `results/` output from `01_full_run` as input datasets, then run.No GPU needed — set Accelerator to **None** so you do not burn quota.There is no fallback data anywhere. If a file is missing this crashes, on purpose.

In [ ]:
import os, sys, glob, shutil, subprocessWORK = "/kaggle/working"; os.chdir(WORK)GITHUB_REPO = ""for src in glob.glob("/kaggle/input/**/*.py", recursive=True): shutil.copy(src, WORK)if not os.path.exists("analyze_and_plot.py") and GITHUB_REPO:    subprocess.run(["git","clone","--depth","1",GITHUB_REPO,"/tmp/kit"], check=True)    for src in glob.glob("/tmp/kit/**/*.py", recursive=True): shutil.copy(src, WORK)os.makedirs("results", exist_ok=True)found = glob.glob("/kaggle/input/**/results/*_v2_results.json", recursive=True) + \        glob.glob("/kaggle/input/*/*_v2_results.json")for f in found: shutil.copy(f, "results/")files = sorted(glob.glob("results/*_v2_results.json"))print(f"{len(files)} results files:")for f in files: print("  ", f)if not files:    raise SystemExit("No results files. Attach the 01_full_run outputs as inputs.")

In [ ]:
!pip -q install -U matplotlib 2>/dev/null | tail -1!python analyze_and_plot.py results/*_v2_results.json --persona default \    --fig-out Figure1.pdf --table-out Table1.txt

In [ ]:
from IPython.display import Image, display!python -c "import fitz" 2>/dev/null || pip -q install pymupdfimport fitzdoc = fitz.open("Figure1.pdf")pix = doc[0].get_pixmap(dpi=140); pix.save("Figure1_preview.png")display(Image("Figure1_preview.png"))

## The scale trend — the figure that may matter more than Figure 1If you ran the ladder, plot Δ against parameter count. A monotone trend is a much strongerclaim than any single point, and it is the finding the funding plan promises for month six.

In [ ]:
import json, glob, reimport numpy as np, matplotlib.pyplot as pltSIZES = {"0.5B":0.5,"1.5B":1.5,"3B":3.0,"7B":7.0,"14B":14.0,"8B":8.0,"2B":2.0}def params(mid):    for k, v in SIZES.items():        if k.lower() in mid.lower(): return v    return Nonepts = []for f in sorted(glob.glob("results/*_v2_results.json")):    r = json.load(open(f)); p = params(r["model_id"])    if p is None: continue    pdata = r["personas"].get("default")    if not pdata: continue    best = None    for dk, e in pdata.items():        if e["ceilings"]["self"]["ceiling_spearman_brown"] >= 0.85:            if best is None or e["delta_self_vs_other"]["delta_mean"] > best[1]["delta_self_vs_other"]["delta_mean"]:                best = (dk, e)    if best is None:         print(f"  skipped {r['model_id']}: no depth cleared the ceiling gate"); continue    e = best[1]    pts.append((p, r["model_id"].split("/")[-1],                e["delta_self_vs_other"]["delta_mean"],                e["delta_self_vs_other"]["ci95"]))if len(pts) < 2:    print("Need >=2 models with a passing ceiling to plot a trend.")else:    pts.sort()    x  = [q[0] for q in pts]; y = [q[2] for q in pts]    lo = [q[2]-q[3][0] for q in pts]; hi = [q[3][1]-q[2] for q in pts]    fig, ax = plt.subplots(figsize=(6,4))    ax.errorbar(x, y, yerr=[lo,hi], marker="o", capsize=4, linewidth=2, color="#1f77b4")    ax.axhline(0, color="black", lw=1)    ax.set_xscale("log"); ax.set_xticks(x); ax.set_xticklabels([f"{q[0]:g}B" for q in pts])    ax.set_xlabel("parameters"); ax.set_ylabel(r"$\Delta$ (self-specificity)")    ax.set_title("Does referential specificity emerge with scale?", fontweight="bold")    fig.tight_layout(); fig.savefig("Figure2_scale.pdf", bbox_inches="tight")    print("wrote Figure2_scale.pdf")    for q in pts: print(f"  {q[1]:32s} delta {q[2]:+.3f} [{q[3][0]:+.3f},{q[3][1]:+.3f}]")

In [ ]:
print(open("Table1.txt").read())

## Before you writeRe-read the sanity-check block printed by `analyze_and_plot.py`. It flags the three things ajudge checks first: gate-failing cells, underpowered nulls being read as trends, and a distressvector that is really a sentiment direction. Report every flag it raises — showing a failedcell reads as rigour, hiding one reads as concealment.